# Download Depth Anything V2 (PyTorch .pth format)

Depth Anything V2 is a monocular depth estimation model. Its weights are distributed as PyTorch `.pth` checkpoint files. This notebook clones the code, downloads the weights, and runs a test. It works on Mac and Windows.

## Which model size?

| Model | Encoder flag | File name | Params | License |
|---|---|---|---|---|
| Small | `vits` | `depth_anything_v2_vits.pth` | 24.8M | Apache-2.0 |
| Base | `vitb` | `depth_anything_v2_vitb.pth` | 97.5M | CC-BY-NC-4.0 |
| Large | `vitl` | `depth_anything_v2_vitl.pth` | 335.3M | CC-BY-NC-4.0 |
| Giant | `vitg` | not released yet ("Coming soon" in the official README) | 1.3B | CC-BY-NC-4.0 |

Start with Small if you just want to test. Only Small allows commercial use.

## Requirements

- Python 3 (the repo does not state a minimum version) and Git
- A GPU is optional. CPU works but is slow. Apple Silicon uses `mps`; Windows uses an NVIDIA GPU via CUDA.
- **Windows with an NVIDIA GPU:** `pip install torch` installs a CPU-only build on Windows. Get the CUDA install command from https://pytorch.org/get-started/locally/ and run it before Step 2.

## Step 1: Clone the repo

In [ ]:
!git clone https://github.com/DepthAnything/Depth-Anything-V2
%cd Depth-Anything-V2

## Step 2: Install dependencies

`%pip` installs into the same environment the notebook kernel uses. Restart the kernel afterwards if the notebook asks you to.

In [ ]:
%pip install -r requirements.txt

## Step 3: Download the checkpoint (.pth)

Set `ENCODER` to the size you want (`vits`, `vitb` or `vitl`). The cell downloads the file into `checkpoints/`.

In [ ]:
import os
import urllib.request

ENCODER = 'vits'  # 'vits' = Small, 'vitb' = Base, 'vitl' = Large

NAMES = {'vits': 'Small', 'vitb': 'Base', 'vitl': 'Large'}
filename = f'depth_anything_v2_{ENCODER}.pth'
url = f'https://huggingface.co/depth-anything/Depth-Anything-V2-{NAMES[ENCODER]}/resolve/main/{filename}?download=true'

os.makedirs('checkpoints', exist_ok=True)
path = os.path.join('checkpoints', filename)
if not os.path.exists(path):
    urllib.request.urlretrieve(url, path)
print(path, f'{os.path.getsize(path) / 1e6:.1f} MB')

You can also download manually from the model pages and move the file into `checkpoints/`:

- Small: https://huggingface.co/depth-anything/Depth-Anything-V2-Small
- Base: https://huggingface.co/depth-anything/Depth-Anything-V2-Base
- Large: https://huggingface.co/depth-anything/Depth-Anything-V2-Large

Or use the Hugging Face CLI (`pip install -U huggingface_hub`, then `hf download depth-anything/Depth-Anything-V2-Small depth_anything_v2_vits.pth --local-dir checkpoints`). Older versions use `huggingface-cli download` with the same arguments.

## Step 4: Run the model in Python

In [ ]:
import glob
import cv2
import torch
import matplotlib.pyplot as plt
from depth_anything_v2.dpt import DepthAnythingV2

DEVICE = 'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'

model_configs = {
    'vits': {'encoder': 'vits', 'features': 64,  'out_channels': [48, 96, 192, 384]},
    'vitb': {'encoder': 'vitb', 'features': 128, 'out_channels': [96, 192, 384, 768]},
    'vitl': {'encoder': 'vitl', 'features': 256, 'out_channels': [256, 512, 1024, 1024]},
}

model = DepthAnythingV2(**model_configs[ENCODER])
model.load_state_dict(torch.load(f'checkpoints/depth_anything_v2_{ENCODER}.pth', map_location='cpu'))
model = model.to(DEVICE).eval()

# Uses an example image from the repo. Replace with your own path.
img_path = sorted(glob.glob('assets/examples/*.jpg'))[0]
raw_img = cv2.imread(img_path)
depth = model.infer_image(raw_img)  # HxW numpy array

fig, ax = plt.subplots(1, 2, figsize=(12, 5))
ax[0].imshow(cv2.cvtColor(raw_img, cv2.COLOR_BGR2RGB)); ax[0].set_title('Input'); ax[0].axis('off')
ax[1].imshow(depth, cmap='inferno'); ax[1].set_title('Depth'); ax[1].axis('off')
plt.show()

## Optional: Run the repo's command-line script

Depth maps are saved to `depth_vis/`.

In [ ]:
!python run.py --encoder {ENCODER} --img-path assets/examples --outdir depth_vis

## Troubleshooting

- **`FileNotFoundError` on the .pth:** the path or file name is wrong. It must match `checkpoints/depth_anything_v2_<encoder>.pth`.
- **Size mismatch when loading:** the `ENCODER` or config does not match the checkpoint (for example, a `vitl` file loaded with `vits`).
- **Out of memory:** use a smaller model or lower the input size.
- **Download is only a few KB:** you saved an HTML page. Use the `resolve/main/...` link, not the `blob/main/...` link.

## Other formats

If you need something other than `.pth` (ONNX, Core ML, TensorRT), export from the PyTorch model after Step 4. Pre-converted versions also exist on Hugging Face (search "Depth Anything V2 ONNX" or "Core ML").

## Links

- Code: https://github.com/DepthAnything/Depth-Anything-V2
- Models: https://huggingface.co/depth-anything